# NB_patient_360

Builds `adb_rapollocare.gold.patient_360` (one row per patient) from the silver layer per the Silver-to-Gold STTM. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance providers and lab results aggregated per patient. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE OR REPLACE TABLE adb_rapollocare.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary. Built by NB_patient_360 (FULL rebuild).'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    count(*)                                AS total_tests,
    count_if(result_status = 'Abnormal')    AS abnormal_test_count,
    count_if(result_status = 'Critical')    AS critical_test_count,
    max(test_date)                          AS last_test_date
  FROM adb_rapollocare.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                             AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                  AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                                          AS insurance_provider_name,
  CAST(coalesce(l.total_tests, 0) AS INT)                                   AS total_tests,
  CAST(coalesce(l.abnormal_test_count, 0) AS INT)                           AS abnormal_test_count,
  CAST(coalesce(l.critical_test_count, 0) AS INT)                           AS critical_test_count,
  l.last_test_date
FROM adb_rapollocare.silver.patients p
LEFT JOIN adb_rapollocare.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_rapollocare.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
# Return the gold table record count to the caller (job / dbutils.notebook.run)
updated_df = spark.table("adb_rapollocare.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))